# QLoRA — Thai slip extraction (Qwen2.5-1.5B-Instruct)

Trains a LoRA adapter on a 4-bit base model (that is what QLoRA is: a frozen
quantised base, a small adapter trained in fp16) and evaluates the **same base
model before and after** on the same 250 held-out slips, on this same GPU.

**Runtime → Change runtime type → T4 GPU** before running anything.

The notebook only produces predictions. Scoring happens back in the repo with
`uv run python -m slipft.score`, so every configuration — base, tuned, and the
API models — is graded by one scorer rather than three.

In [ ]:
!nvidia-smi -L
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

In [ ]:
%%capture
# Unsloth pulls a matching torch / bitsandbytes / trl / peft set for the runtime.
!pip install -q unsloth
!pip install -q --force-reinstall --no-cache-dir --no-deps git+https://github.com/unslothai/unsloth.git

## Data

Set `REPO_URL` to the GitHub repo to pull `data/*.jsonl` from. Leave it empty
to upload the three files by hand instead.

In [ ]:
REPO_URL = ""  # e.g. "https://github.com/annop07/thai-slip-qlora.git"

import os, json, pathlib

if REPO_URL:
    !git clone -q $REPO_URL repo
    DATA = pathlib.Path('repo/data')
else:
    from google.colab import files
    DATA = pathlib.Path('data'); DATA.mkdir(exist_ok=True)
    print('upload train.jsonl, valid.jsonl and test.jsonl')
    for name, blob in files.upload().items():
        (DATA / name).write_bytes(blob)

def read_jsonl(path):
    with open(path, encoding='utf-8') as f:
        return [json.loads(line) for line in f]

train_rows = read_jsonl(DATA / 'train.jsonl')
valid_rows = read_jsonl(DATA / 'valid.jsonl')
test_rows  = read_jsonl(DATA / 'test.jsonl')
len(train_rows), len(valid_rows), len(test_rows)

## Load the base model in 4-bit

`max_seq_length` is 1024 because the longest slip plus its JSON answer fits in
about 600 tokens — a longer window would cost memory and buy nothing.

In [ ]:
from unsloth import FastLanguageModel
import torch

MODEL = 'unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit'
MAX_SEQ = 1024

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = MODEL,
    max_seq_length = MAX_SEQ,
    load_in_4bit = True,   # NF4 — the Q of QLoRA
    dtype = None,          # fp16 on a T4, bf16 where it is supported
)
print(model.config.torch_dtype, next(model.parameters()).device)

## The evaluation loop

Greedy decoding, one slip at a time. Batching would be faster and would make
the latency column meaningless — per-request latency is the number a service
actually has to meet.

In [ ]:
import time

SYSTEM = json.loads(open(DATA / 'train.jsonl', encoding='utf-8').readline())['messages'][0]['content']
print(SYSTEM[:120], '...')

def build_prompt(slip_text):
    messages = [
        {'role': 'system', 'content': SYSTEM},
        {'role': 'user', 'content': f'Slip text:\n{slip_text}'},
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

@torch.inference_mode()
def run_eval(tag, rows, max_new_tokens=320):
    FastLanguageModel.for_inference(model)
    out_path = f'{tag}.predictions.jsonl'
    with open(out_path, 'w', encoding='utf-8') as f:
        for i, row in enumerate(rows):
            prompt = build_prompt(row['text'])
            inputs = tokenizer(prompt, return_tensors='pt').to('cuda')
            started = time.perf_counter()
            generated = model.generate(
                **inputs,
                max_new_tokens = max_new_tokens,
                do_sample = False,              # greedy: the run must be repeatable
                pad_token_id = tokenizer.eos_token_id,
            )
            latency_ms = (time.perf_counter() - started) * 1000
            completion = generated[0][inputs['input_ids'].shape[1]:]
            f.write(json.dumps({
                'id': row['id'],
                'model': f'{MODEL} ({tag})',
                'output': tokenizer.decode(completion, skip_special_tokens=True).strip(),
                'latency_ms': latency_ms,
                'prompt_tokens': int(inputs['input_ids'].shape[1]),
                'completion_tokens': int(completion.shape[0]),
            }, ensure_ascii=False) + '\n')
            if (i + 1) % 25 == 0:
                print(f'  {tag}: {i + 1}/{len(rows)}')
    print('wrote', out_path)
    return out_path

## Base model, before any training

Same weights, same prompt, same GPU as the run after training. This cell is
the control, and it has to run first — once the adapter is attached there is
no untouched model left to measure.

In [ ]:
base_started = time.perf_counter()
run_eval('base', test_rows)
print(f'base eval took {(time.perf_counter() - base_started) / 60:.1f} min')

## Attach the LoRA adapter and train

`r=16` on the attention and MLP projections is ~18M trainable parameters out
of 1.5B — a bit over 1%. The base stays frozen and quantised throughout.

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    lora_alpha = 16,
    lora_dropout = 0,
    bias = 'none',
    target_modules = ['q_proj', 'k_proj', 'v_proj', 'o_proj',
                      'gate_proj', 'up_proj', 'down_proj'],
    use_gradient_checkpointing = 'unsloth',
    random_state = 20260819,
)
model.print_trainable_parameters()

In [ ]:
from datasets import Dataset

def to_text(rows):
    return Dataset.from_list([
        {'text': tokenizer.apply_chat_template(r['messages'], tokenize=False)}
        for r in rows
    ])

train_ds, valid_ds = to_text(train_rows), to_text(valid_rows)
print(train_ds[0]['text'][:400])

In [ ]:
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_ds,
    eval_dataset = valid_ds,
    args = SFTConfig(
        dataset_text_field = 'text',
        max_seq_length = MAX_SEQ,
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,   # effective batch 8
        num_train_epochs = 2,
        learning_rate = 2e-4,
        warmup_ratio = 0.05,
        lr_scheduler_type = 'cosine',
        logging_steps = 10,
        eval_strategy = 'epoch',
        optim = 'adamw_8bit',
        weight_decay = 0.01,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        seed = 20260819,
        output_dir = 'outputs',
        report_to = 'none',
    ),
)

# Loss on the answer only. Without this the model spends most of its gradient
# budget learning to reproduce the system prompt it is always given anyway.
trainer = train_on_responses_only(
    trainer,
    instruction_part = '<|im_start|>user\n',
    response_part = '<|im_start|>assistant\n',
)

In [ ]:
train_started = time.perf_counter()
stats = trainer.train()
train_minutes = (time.perf_counter() - train_started) / 60
peak_gb = torch.cuda.max_memory_reserved() / 1024**3
print(f'{train_minutes:.1f} min, peak {peak_gb:.1f} GB')
print(stats.metrics)

## The same model, after training

In [ ]:
run_eval('tuned', test_rows)

## Save everything worth keeping

The adapter is ~40 MB — the 4-bit base it attaches to is downloaded from the
Hub, so this is the entire artifact of the training run.

In [ ]:
import shutil

model.save_pretrained('adapter')
tokenizer.save_pretrained('adapter')
shutil.make_archive('adapter', 'zip', 'adapter')

with open('train_log.json', 'w') as f:
    json.dump({
        'base_model': MODEL,
        'gpu': torch.cuda.get_device_name(0),
        'train_rows': len(train_rows),
        'epochs': 2,
        'effective_batch': 8,
        'learning_rate': 2e-4,
        'lora_r': 16,
        'train_minutes': train_minutes,
        'peak_memory_gb': peak_gb,
        'metrics': stats.metrics,
        'log_history': trainer.state.log_history,
    }, f, indent=2)

from google.colab import files
for name in ['base.predictions.jsonl', 'tuned.predictions.jsonl', 'train_log.json', 'adapter.zip']:
    files.download(name)

## Back in the repo

```bash
mv ~/Downloads/base.predictions.jsonl  results/
mv ~/Downloads/tuned.predictions.jsonl results/
mv ~/Downloads/train_log.json          results/

uv run python -m slipft.score results/*.predictions.jsonl
```

If the Unsloth install fails on a future Colab image, the same run works with
stock libraries — `BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4',
bnb_4bit_compute_dtype=torch.float16)` on `AutoModelForCausalLM`, then
`peft.get_peft_model` with the same LoRA config and the same `SFTTrainer`. Only
the first two cells change; it trains roughly twice as slowly.